# VOD archive: transcription on Colab

GPU stage of the pipeline (voice enrollment and `process.py`). Used instead of `deploy/gpu/run_remote.sh`.

**Once:**
- Runtime → Change runtime type → **T4 GPU** (or better).
- 🔑 Secrets (left sidebar): add `HF_TOKEN` (the Hugging Face token) and allow notebook access.

**Each run:**
1. On the laptop: `deploy/colab/pack.sh process --model large-v3` (includes `voices/`, for `TASK = "both"`), or `deploy/colab/pack.sh enroll`.
2. Upload `colab/bundle.tar` to Google Drive as **My Drive/vod-archive/bundle.tar** (replace the old one).
3. Set `TASK` / `ARGS` below, then Runtime → **Run all**.
4. Download **My Drive/vod-archive/out** from Drive, then on the laptop: `deploy/colab/unpack.sh <the download>`.

Transcripts are written straight to Drive (`vod-archive/out/transcripts`), so if Colab disconnects, run all
again: finished VODs are skipped.

In [ ]:
TASK = "process"            # "enroll", "process", or "both" (enroll, then process with the new voiceprints)
ARGS = "--model large-v3"   # process.py arguments (pilot: run once with large-v3, once with large-v3-turbo)
DRIVE_DIR = "/content/drive/MyDrive/vod-archive"
REPO = "/content/thehasanproject"

In [ ]:
import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.path.exists(f"{DRIVE_DIR}/bundle.tar"), f"upload bundle.tar to {DRIVE_DIR} first"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/carrier3rd/thehasanproject {REPO}
%cd {REPO}
!pip install -q -r requirements-gpu.txt

In [ ]:
# inputs from the bundle; outputs go straight to Drive so a disconnect loses at most the VOD in progress
!tar -xf {DRIVE_DIR}/bundle.tar -C {REPO}
OUT = f"{DRIVE_DIR}/out"
for sub in ("transcripts", "logs", "voiceprints"):
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)
for link, target in (("data/transcripts", f"{OUT}/transcripts"), ("logs", f"{OUT}/logs")):
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.isdir(link):
        !cp -rn {link}/. {target}/ && rm -rf {link}
    os.makedirs(os.path.dirname(link) or ".", exist_ok=True)
    os.symlink(target, link)
os.environ["VOD_ARCHIVE_PRIVATE_DIR"] = "private"
!python check_env.py --role gpu
!ls data/audio voices 2>/dev/null

In [ ]:
assert TASK in ("enroll", "process", "both"), "TASK must be 'enroll', 'process' or 'both'"
if TASK in ("enroll", "both"):
    !python pipeline/enroll_voices.py
    !cp voiceprints/*.npy {OUT}/voiceprints/ && ls -l {OUT}/voiceprints
if TASK in ("process", "both"):
    !python pipeline/process.py {ARGS}
    !find {OUT}/transcripts -name '*.json' | sort

In [ ]:
drive.flush_and_unmount()   # make sure everything reached Drive before you download it
print("Done. Download My Drive/vod-archive/out, then run deploy/colab/unpack.sh on the laptop.")